# Square-QDM evidence for the July 23 ICQMBS draft

This notebook is keyed to **Sec. VII and Apps. D, F** of the current draft. It supplies the finite-size Type-I scorecard, the basis-independent compact/collective decomposition, bounded-versus-collective cancellation radii, and the layered deformation diagnostics requested by the manuscript.

The thermodynamic calculation is deliberately a **fixed-width limit**. We take $L_y=4$ and $L_x=4N\to\infty$ in the same electric-winding sector. The primary thermal comparator is not the beta-zero strip trace. It is an energy-matched, symmetry-resolved microcanonical state of the same finite-width Hamiltonian,

$$
\rho^{\rm mc}_{L_x}(E_{\rm cage})
=\frac{P_{|H-E_{\rm cage}|\le \Delta E_{L_x}}}
{\operatorname{Tr}P_{|H-E_{\rm cage}|\le \Delta E_{L_x}}},
\qquad
\Delta E_{L_x}=cJ\sqrt{4L_x}.
$$

Thus $\Delta E_{L_x}/(4L_x)\to0$, while the retained state count must be checked numerically. The microcanonical state is constructed in the same $(W_x,W_y)$ sector and, for a translation-invariant Peierls-deformed Hamiltonian, in the same $(k_x,k_y)$ sector as the projected cage. Exact degeneracies at the window boundary are retained.

The beta-zero transfer calculation is kept as an analytical strip reference, but it is not substituted for the finite-temperature comparison at the cage energy. A genuine $L_x,L_y\to\infty$ result remains outside the scope of this notebook.

## Imports, reproducibility, and execution controls

The $4\times4$ calculation is inexpensive. The $8\times4$ zero-winding basis has dimension about $1.7\times10^4$, but translation projection reduces the selected momentum sector to roughly $5.7\times10^2$ states; it is enabled by default. Set `RUN_8X4_MICROCANONICAL=False` for a rapid smoke test.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as scipy_linalg
from IPython.display import display

from helpers import save_prx_figure, set_revtex_matplotlib_style

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    LocalQDMCageSearchConfig,
    Quasi1DSequencePoint,
    RobustQDMLocalCageSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    adjacent_gap_ratio_report,
    audit_quasi_1d_sequence,
    beta_zero_matching_subspace,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_finite_size_scorecard,
    cage_jacobian_conditioning_from_hamiltonian,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state,
    commuting_cyclic_symmetry_sector_basis,
    diagnose_boundary_cancellation_matroid,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    eigenstate_expectations,
    evaluate_square_qdm_classification_witnesses_on_strips,
    gaussian_spectral_filter,
    local_witnesses_from_classification_report,
    materialize_square_qdm_periodic_product_state,
    operator_coefficient_compatibility,
    partition_cage_hamiltonian,
    product_basis_diagonal_phase_factors,
    project_coefficients_to_beta_zero_match,
    project_operator_to_sector,
    project_state_to_sector,
    regional_cage_quotient,
    robust_qdm_local_cage_search,
    scan_square_qdm_beta_zero_energy_density,
    scan_square_qdm_collective_locality_extension,
    scan_square_qdm_periodic_product_cancellation_scaling,
    scan_windowed_operator_annihilators,
    select_microcanonical_window_by_count,
    select_microcanonical_window_by_width,
    spectral_observable_moments,
    subspace_complement_basis,
    thermal_activity_margin_from_samples,
    thermodynamic_energy_window_plan,
)
from qlinks.models import (
    SquareQDMModel,
    qdm_peierls_couplings_from_link_phases,
    qdm_plaquette_link_gauge_matrix,
)
from qlinks.operators import PlaquettePatternOperator

TOL = 1.0e-10
RANK_TOL = 1.0e-9
RANDOM_SEED = 73291
USE_TEX = False
SAVE_FIGURES = True
SAVE_PDF = True
RUN_8X4_MICROCANONICAL = True

PEIERLS_REFERENCE_PHASE = 0.35
PEIERLS_PATH = np.linspace(0.15, 0.55, 5)
MICROCANONICAL_PREFACTORS = (0.50, 0.75, 1.00)
PRIMARY_WINDOW_PREFACTOR = 0.75
SMOOTH_SIGMA_PREFACTOR = 0.75

DATA_DIR = REPO_ROOT / "experimental" / "data" / "square_qdm_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

set_revtex_matplotlib_style(base_font_size=9, prefer_tex=USE_TEX)

FIGURE_FORMATS = ("pdf", "svg")

def save_figure(fig, stem, *, aliases=(), close=False):
    save_prx_figure(fig, stem, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    for alias in aliases:
        if alias != stem:
            save_prx_figure(fig, alias, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    if close:
        plt.close(fig)


def embedded_state(record, hilbert_size):
    state = np.zeros(hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return state


def square_qdm_basis_translation_permutation(model, basis_configs, *, dx=0, dy=0):
    """Return the basis permutation for a physical lattice translation."""
    link_lookup = {}
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        link_lookup[(int(x), int(y), str(link.kind))] = int(link.id)
    transformed = np.zeros_like(basis_configs)
    lx, ly = int(model.lattice.lx), int(model.lattice.ly)
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        target = link_lookup[((int(x) + dx) % lx, (int(y) + dy) % ly, str(link.kind))]
        transformed[:, target] = basis_configs[:, int(link.id)]
    lookup = {
        tuple(int(value) for value in config): index
        for index, config in enumerate(basis_configs)
    }
    return np.asarray(
        [lookup[tuple(int(value) for value in config)] for config in transformed],
        dtype=np.int64,
    )


def fixed_width_cage_momentum(repeats):
    """Momentum branch followed by the repeated compact product cage."""
    return 0, (2 * int(repeats)) % 4


print({
    "repository": str(REPO_ROOT),
    "data_directory": str(DATA_DIR),
    "run_8x4_microcanonical": RUN_8X4_MICROCANONICAL,
    "reference_peierls_phase": PEIERLS_REFERENCE_PHASE,
})

## 1. Reconstruct the $4\times4$ Type-I census

The draft asks for the exact sector, shell dimensions, boundary shapes, rank/nullity, residual tolerances, support sizes, and degeneracy handling.  The IPR basis selection is used only after the invariant nullspace has been certified.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)

t0 = time.perf_counter()
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
build_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=TOL,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()
search_seconds = time.perf_counter() - t0

records_04 = tuple(square_search[(0, 4)])
record_06 = square_search[(0, 6), 0]
states_04 = np.column_stack(
    [embedded_state(record, square_search.hilbert_size) for record in records_04]
)

{
    "winding_sector": (0, 0),
    "hilbert_dimension": square_search.hilbert_size,
    "counts_by_signature": square_search.counts_by_signature,
    "build_seconds": build_seconds,
    "search_seconds": search_seconds,
}

In [ ]:
scorecard_rows = []
for signature, records in (((0, 4), records_04), ((0, 6), (record_06,))):
    for record_index, record in enumerate(records):
        full_state = embedded_state(record, square_search.hilbert_size)
        scorecard = cage_finite_size_scorecard(
            square_build.hamiltonian,
            record.candidate.vertices,
            full_state,
            kinetic=square_build.kinetic,
            actual_support=record.cage_state.support,
            amplitude_tolerance=TOL,
            rank_tolerance=TOL,
            metadata={
                "signature": str(signature),
                "record": record_index,
                "basis_strategy": "IPR postselection",
                "winding_x": 0,
                "winding_y": 0,
            },
        )
        scorecard_rows.append(scorecard.to_summary_dict())

scorecard_table = pd.DataFrame(scorecard_rows)
scorecard_table.to_csv(DATA_DIR / "qdm_4x4_type1_scorecard.csv", index=False)
display(scorecard_table[[
    "signature", "record", "candidate_shell_size", "boundary_rows",
    "boundary_columns", "boundary_rank", "boundary_nullity",
    "boundary_singular_gap", "actual_support_size",
    "internal_residual", "boundary_residual", "relative_eigenpair_residual",
]])

The $(0,4)$ shell is a $84\times48$ boundary problem with nullity nine.  Eight IPR representatives have support four, while the ninth has support 48.  The $(0,6)$ shell is a $100\times32$ boundary problem with nullity one.

## 2. Basis-independent $9=8+1$ decomposition

The compact subspace is defined as the span of the eight support-four records.  The quotient of the complete $(0,4)$ cage manifold by this compact span is one dimensional, and its vector has unit overlap with IPR record 8.

In [ ]:
regional_supports = tuple(record.cage_state.support for record in records_04[:8])
quotient_report = regional_cage_quotient(
    square_build.kinetic,
    regional_supports,
    states_04,
    tolerance=TOL,
)
quotient_overlaps = np.abs(states_04.conj().T @ quotient_report.quotient_basis) ** 2

full_support = tuple(
    sorted(set().union(*(set(record.cage_state.support) for record in records_04)))
)
full_blocks = partition_cage_hamiltonian(square_build.kinetic, full_support)
full_column = {basis_index: column for column, basis_index in enumerate(full_support)}
regional_columns = tuple(
    tuple(full_column[basis_index] for basis_index in support)
    for support in regional_supports
)
matroid_report = diagnose_boundary_cancellation_matroid(
    full_blocks.boundary,
    regional_columns,
    tolerance=TOL,
)

quotient_table = pd.DataFrame([{
    **quotient_report.to_summary_dict(),
    "collective_record_overlap": float(quotient_overlaps[8, 0]),
    "regional_circuit_count": matroid_report.regional_circuit_count,
    "weighted_relative_dependency_dimension": matroid_report.relative_dependency_dimension,
}])
quotient_table.to_csv(DATA_DIR / "qdm_4x4_compact_collective_quotient.csv", index=False)
display(quotient_table.T)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.bar(["complete cage\nmanifold", "compact span", "collective\nquotient"], [9, 8, 1])
ax.set_ylabel("Dimension")
ax.set_ylim(0, 10)
ax.set_title(r"$(0,4)$ cage-space decomposition")
save_figure(fig, "qdm_4x4_9_equals_8_plus_1")
plt.show()

## 3. Minimal local cancellation radius

For each plaquette kinetic term $K_p$, form the action vector $K_p|\psi\rangle$.  Inside each periodic real-space window, minimize the action norm over coefficient vectors of unit Euclidean norm.  The resulting smallest singular value is a coefficient-normalized annihilation residual.

This is a direct numerical implementation of the draft note requesting the residual versus allowed real-space radius.  It does not assume that the optimal coefficients are pairwise or equal-weight.

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.potential_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

radius_scans = {}
for label, state in (
    ("compact record 0", states_04[:, 0]),
    ("collective record 8", states_04[:, 8]),
):
    radius_scans[label] = scan_windowed_operator_annihilators(
        kinetic_term_matrices,
        state,
        plaquette_centers,
        radii=(0, 1, 2),
        periodic_box=(4, 4),
        metric="chebyshev",
        normalize_actions=True,
        action_tolerance=1.0e-12,
        rank_tolerance=TOL,
    )

radius_rows = [
    {"state": label, **point.to_summary_dict()}
    for label, report in radius_scans.items()
    for point in report.points
]
radius_table = pd.DataFrame(radius_rows)
radius_table.to_csv(DATA_DIR / "qdm_4x4_minimum_annihilator_radius.csv", index=False)
display(radius_table[[
    "state", "radius", "minimum_residual", "n_active", "rank", "nullity",
    "coefficient_support_size", "active_operator_indices",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
for label, report in radius_scans.items():
    ax.semilogy(
        [point.radius for point in report.points],
        [max(point.minimum_residual, 1.0e-16) for point in report.points],
        marker="o",
        label=label,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="numerical tolerance")
ax.set_xlabel("Allowed Chebyshev radius")
ax.set_ylabel("Minimum annihilation residual")
ax.set_xticks([0, 1, 2])
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_annihilator_radius")
plt.show()

The compact representative first reaches the numerical kernel at radius one using two active plaquette terms.  The collective quotient retains a residual about $0.675$ at radius one and reaches a kernel only when all 16 plaquette terms are available.  On the $4\times4$ torus, this is evidence for a system-scale cancellation, not a proof that its radius must diverge on every possible continuation.

## 4. Layered deformation diagnostics, including Peierls phases

The draft distinguishes exact fixed-state compatibility from first-order continuation of a nearby cage. We therefore evaluate the full cage-obstruction hierarchy for four declared real parameter alphabets:

1. independent plaquette-flip amplitudes;
2. independent infinitesimal Peierls phases;
3. independent plaquette flippability potentials;
4. the combined amplitude-plus-phase alphabet.

For a plaquette phase $\phi_p$, the tangent at $\phi_p=0$ is $iU_p-iU_p^\dagger$. All coefficient vectors below are real; the perturbation matrices may be complex Hermitian.

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.potential_operators
)
if len(potential_term_matrices) != 2 * len(square_model.plaquette_ids()):
    raise RuntimeError("Expected two orientation projectors per square plaquette.")
plaquette_potential_matrices = tuple(
    potential_term_matrices[2 * index] + potential_term_matrices[2 * index + 1]
    for index in range(len(square_model.plaquette_ids()))
)
phase_tangent_operators = tuple(
    PlaquettePatternOperator.qdm_flip(
        layout=square_model.layout,
        lattice=square_model.lattice,
        plaquette_id=int(plaquette_id),
        coefficient=1.0j,
        reverse_coefficient=-1.0j,
    )
    for plaquette_id in square_model.plaquette_ids()
)
phase_tangent_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in phase_tangent_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

compact_state = states_04[:, 0]
collective_state = states_04[:, 8]
deformation_alphabets = {
    "flip amplitudes": kinetic_term_matrices,
    "Peierls phases": phase_tangent_matrices,
    "flippability potentials": plaquette_potential_matrices,
    "amplitudes + phases": kinetic_term_matrices + phase_tangent_matrices,
}
state_targets = {
    "compact": compact_state,
    "collective": collective_state,
}

hierarchy_reports = {}
hierarchy_rows = []
singular_rows = []
conditioning_rows = []
for target_name, state in state_targets.items():
    support = np.flatnonzero(np.abs(state) > TOL)
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        square_build.hamiltonian,
        support,
        state,
        tolerance=TOL,
    )
    conditioning_rows.append({
        "target": target_name,
        **conditioning.to_summary_dict(),
    })
    for alphabet_name, perturbations in deformation_alphabets.items():
        report = cage_compatibility_hierarchy_from_hamiltonians(
            square_build.hamiltonian,
            perturbations,
            support,
            state,
            coefficient_field="real",
            tolerance=TOL,
        )
        hierarchy_reports[(target_name, alphabet_name)] = report
        hierarchy_rows.append({
            "target": target_name,
            "alphabet": alphabet_name,
            "obstruction_rank": report.first_order.rank,
            **report.to_summary_dict(),
        })
        singular_rows.extend(
            {
                "target": target_name,
                "alphabet": alphabet_name,
                "singular_index": singular_index,
                "singular_value": float(singular_value),
            }
            for singular_index, singular_value in enumerate(report.first_order.singular_values)
        )

hierarchy_table = pd.DataFrame(hierarchy_rows)
conditioning_table = pd.DataFrame(conditioning_rows)
singular_table = pd.DataFrame(singular_rows)
hierarchy_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_hierarchy.csv", index=False)
conditioning_table.to_csv(DATA_DIR / "qdm_4x4_cage_conditioning.csv", index=False)
singular_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_spectra.csv", index=False)
display(hierarchy_table)
display(conditioning_table[["target", "support_size", "cage_gap", "full_residual"]])

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.2))
plot_table = hierarchy_table.copy()
x = np.arange(len(deformation_alphabets))
width = 0.36
for target_index, target_name in enumerate(("compact", "collective")):
    group = plot_table[plot_table["target"] == target_name].set_index("alphabet").loc[list(deformation_alphabets)]
    ax.bar(
        x + (target_index - 0.5) * width,
        group["first_order_compatible_dimension"],
        width=width,
        label=target_name,
    )
ax.set_xticks(x, list(deformation_alphabets), rotation=16, ha="right")
ax.set_ylabel("First-order compatible dimension")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_deformation_compatible_dimensions")
plt.show()

fig, ax = plt.subplots(figsize=(5.8, 3.2))
for (target_name, alphabet_name), group in singular_table.groupby(["target", "alphabet"], sort=False):
    if alphabet_name not in ("flip amplitudes", "Peierls phases"):
        continue
    ax.semilogy(
        group["singular_index"] + 1,
        np.maximum(group["singular_value"], 1.0e-16),
        marker="o",
        label=f"{target_name}: {alphabet_name}",
    )
ax.set_xlabel("Obstruction singular-value index")
ax.set_ylabel("Singular value")
ax.legend(frameon=False, fontsize=7)
save_figure(fig, "qdm_4x4_deformation_obstruction_spectra")
plt.show()

### Uniform Peierls-flux path and the gauge-generated subspace

A link-diagonal unitary $\exp(i\sum_\ell\theta_\ell n_\ell)$ generates plaquette phases in a rank-15 subspace on the $4\times4$ torus. The single phase pattern orthogonal to this subspace is the uniform plaquette phase. It is therefore a useful physical deformation rather than a mere basis-gauge change.

The compact representative remains an exact cage along this path. The collective quotient is lifted immediately. This supplies a visually direct deformation contrast complementary to the tangent-space ranks.

In [ ]:
gauge_incidence = qdm_plaquette_link_gauge_matrix(square_model.lattice)
gauge_rank = int(np.linalg.matrix_rank(gauge_incidence, tol=RANK_TOL))
uniform_phase_pattern = np.ones(len(square_model.plaquette_ids()), dtype=np.float64)
gauge_projection = gauge_incidence @ np.linalg.lstsq(
    gauge_incidence,
    uniform_phase_pattern,
    rcond=RANK_TOL,
)[0]
uniform_non_gauge_residual = float(np.linalg.norm(uniform_phase_pattern - gauge_projection))

peierls_rows = []
for phase in np.linspace(-0.70, 0.70, 15):
    phase_model = replace(square_model, coup_kin=np.exp(1.0j * phase))
    phase_build = phase_model.build(
        basis_solver="dfs",
        builder="sparse",
        backend="scipy",
        sort_basis=True,
    )
    np.testing.assert_array_equal(phase_build.basis.states, square_build.basis.states)
    for target_name, state in state_targets.items():
        support = np.flatnonzero(np.abs(state) > TOL)
        eigenpair = diagnose_eigenpair(phase_build.hamiltonian, state)
        conditioning = cage_jacobian_conditioning_from_hamiltonian(
            phase_build.hamiltonian,
            support,
            state,
            tolerance=TOL,
        )
        peierls_rows.append({
            "phase": float(phase),
            "target": target_name,
            "energy": float(eigenpair.energy.real),
            "residual": eigenpair.residual_norm,
            "Delta_cage": conditioning.cage_gap,
        })

peierls_path_table = pd.DataFrame(peierls_rows)
peierls_path_table.to_csv(DATA_DIR / "qdm_4x4_uniform_peierls_path.csv", index=False)
pd.DataFrame([{
    "n_plaquette_phases": gauge_incidence.shape[0],
    "n_link_phases": gauge_incidence.shape[1],
    "link_gauge_phase_rank": gauge_rank,
    "uniform_phase_distance_from_link_gauge_subspace": uniform_non_gauge_residual,
}]).to_csv(DATA_DIR / "qdm_4x4_peierls_gauge_rank.csv", index=False)
display(peierls_path_table)

fig, ax = plt.subplots(figsize=(3.5, 2.55))
for target_name, group in peierls_path_table.groupby("target"):
    ax.semilogy(
        group["phase"],
        np.maximum(group["residual"], 1.0e-16),
        marker="o",
        label=target_name,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="tolerance")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Fixed-vector residual")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_compact_collective_residual")
plt.show()

fig, ax = plt.subplots(figsize=(3.5, 2.55))
for target_name, group in peierls_path_table.groupby("target"):
    ax.plot(group["phase"], group["Delta_cage"], marker="o", label=target_name)
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel(r"Cage-conditioning gap $\Delta_{\rm cage}$")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_cage_gap")
plt.show()

The rank data and the nonlinear phase path answer different questions. Peierls-phase tangents allow the compact state to continue throughout the complete 16-dimensional phase alphabet at first order. The collective state has one first-order phase obstruction, while the remaining directions mostly rotate its amplitudes rather than preserving the original vector. Along the uniform non-gauge phase, the compact state stays exact but the collective quotient acquires a linear residual.

## 5. Exact $4N\times4$ product sequence and its Peierls continuation

The robust local search reconstructs two independently caged stripe blocks in the $4\times4$ unit cell. We repeat the certified cell along $x$, producing $(4N)\times4$. The local-action proof holds for every positive repeat count. We also recertify the same sequence at the reference Peierls phase $\phi=0.35$ used for the finite-width microcanonical calculation.

In [ ]:
local_search_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    prune_inactive_local_basis_states=True,
    tolerance=TOL,
    degenerate_basis_strategy="ipr",
    ipr_random_seed=1234,
)
robust_config = RobustQDMLocalCageSearchConfig(
    local_config=local_search_config,
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)

stripe_certified, stripe_context = robust_qdm_local_cage_search(
    square_model,
    config=robust_config,
    return_context=True,
)

repeatable_candidates = []
for report_index, report in enumerate(stripe_certified.reports):
    try:
        candidate = SquareQDMPeriodicProductUnitCell.from_padding(
            square_model,
            stripe_context.blocks,
            report.padding,
            repeat_axis="x",
        )
        certificate = certify_square_qdm_periodic_product_sequence(candidate)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable_candidates.append((report_index, candidate, certificate))

if not repeatable_candidates:
    raise RuntimeError("No x-repeatable square-QDM product unit cell was found.")

repeatable_report_index, product_unit_cell, product_sequence = repeatable_candidates[0]
{
    "is_certified": product_sequence.is_certified,
    "repeat_axis": product_unit_cell.repeat_axis,
    "energy_density": product_sequence.energy_density,
    "support_size_per_unit_cell": product_unit_cell.support_size_per_unit_cell,
    "unit_cell_winding_sector": product_sequence.unit_cell_winding_sector,
    "verification_repeats": product_sequence.verification_repeats,
}

peierls_product_unit_cell = product_unit_cell.with_couplings(
    coup_kin=np.exp(1.0j * PEIERLS_REFERENCE_PHASE),
    coup_pot=1.0,
)
peierls_product_sequence = certify_square_qdm_periodic_product_sequence(
    peierls_product_unit_cell,
    tolerance=1.0e-9,
)
if not peierls_product_sequence.is_certified:
    raise RuntimeError("The reference Peierls sequence failed exact certification.")
print({
    "peierls_phase": PEIERLS_REFERENCE_PHASE,
    "peierls_sequence_certified": peierls_product_sequence.is_certified,
    "peierls_energy_density": peierls_product_sequence.energy_density,
})


In [ ]:
product_scaling = scan_square_qdm_periodic_product_cancellation_scaling(
    product_unit_cell,
    repeat_counts=(1, 2, 3),
    max_support_size=128,
    tolerance=1.0e-9,
)
product_scaling_table = pd.DataFrame([
    point.to_summary_dict() for point in product_scaling.points
])
product_scaling_table.to_csv(DATA_DIR / "qdm_4N_by_4_exact_sequence.csv", index=False)
display(product_scaling_table[[
    "repeats", "system_size", "support_size", "shell_size",
    "boundary_nullity", "interference_gap", "product_state_boundary_residual",
    "kinetic_constraint_rank", "kinetic_compatible_dimension",
    "kinetic_compatible_fraction", "potential_constraint_rank",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["interference_gap"], marker="o")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel(r"Boundary singular gap $\Delta_B$")
ax.set_xticks(product_scaling_table["repeats"])
save_figure(fig, "qdm_strip_interference_gap")
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["kinetic_constraint_rank"], marker="o", label="compatibility rank")
ax.plot(product_scaling_table["repeats"], [16*n for n in product_scaling_table["repeats"]], marker="o", label="local kinetic parameters")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel("Dimension")
ax.set_xticks(product_scaling_table["repeats"])
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_compatibility_scaling")
plt.show()

The exact product sequence has support $4^N$, boundary nullity one, and interference gap two on the explicit sizes.  Its preserving kinetic constraints have rank $2N$ out of $16N$ plaquette couplings.  Therefore the exact sequence is size extensible but its generic local-deformation compatibility has an **extensive codimension**.  This is not finite-codimension topological protection.

## 6. Fixed local witness and the beta-zero strip reference

A compact stripe record supplies one Hermitian reduced-IZ witness. The same normalized local operator annihilates the repeated cage and its reference Peierls continuation. The transfer calculation evaluates its exact $\beta=0$ activity in the $(W_x,W_y)=(0,0)$ width-four strip sector without diagonalizing the exponentially growing strip Hilbert space.

This is an analytical reference, not the primary thermal comparator: for $\lambda=1$, the cage energy density differs from the strip's beta-zero energy density.

In [ ]:
stripe_record = stripe_certified.records[repeatable_report_index]
stripe_classification = classify_cage_state(
    stripe_record.cage_state,
    kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,
    hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
stripe_witnesses = local_witnesses_from_classification_report(stripe_classification)
sequence_witness = certify_local_witness_on_square_qdm_periodic_sequence(
    product_sequence,
    stripe_witnesses[0],
    normalization="operator_norm",
)

strip_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128)
strip_witness_report = evaluate_square_qdm_classification_witnesses_on_strips(
    stripe_classification,
    model=square_model,
    lengths=strip_lengths,
    winding_sector=(0, 0),
    normalization="operator_norm",
    winding_projection="fourier",
)
selected_strip_witness = strip_witness_report.records[0]

strip_witness_table = pd.DataFrame([
    evaluation.to_summary_dict()
    for evaluation in selected_strip_witness.scaling_report.evaluations
])
strip_witness_table["cage_expectation"] = 0.0
strip_witness_table.to_csv(DATA_DIR / "qdm_strip_witness_beta_zero.csv", index=False)

{
    "sequence_witness": sequence_witness.to_summary_dict(),
    "n_available_witnesses": len(strip_witness_report.records),
    "selected_window_width": selected_strip_witness.placement.window_width,
    "selected_link_coordinates": selected_strip_witness.placement.link_coordinates,
    "tail_estimate": selected_strip_witness.scaling_report.tail_estimate(),
}

peierls_sequence_witness = certify_local_witness_on_square_qdm_periodic_sequence(
    peierls_product_sequence,
    stripe_witnesses[0],
    normalization="operator_norm",
)
local_q_spectrum = diagnose_local_channel_spectrum(
    sequence_witness.witness,
    tolerance=TOL,
)
print({
    "Delta_Q": local_q_spectrum.dark_channel_gap,
    "Peierls_witness_residual": peierls_sequence_witness.annihilation_residual,
})


In [ ]:
display(strip_witness_table[[
    "length", "circumference", "expectation", "cage_expectation",
    "partition_count", "window_width",
]])

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(strip_witness_table["length"], strip_witness_table["expectation"], marker="o", label=r"$\beta=0$, $(0,0)$ sector")
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel(r"$\mathrm{Tr}(\rho_{\beta=0}Q_R)$")
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_witness_activity")
plt.show()

The beta-zero activity remains positive and approaches a nonzero strip value. It shows that the channel is not kinematically forbidden by the winding constraint, but it does **not** establish the thermal activity at the cage energy of the uniform $\lambda=1$ model. The energy-matched comparison follows next.

## 7. Energy-matched fixed-width microcanonical ensemble

For each repeated cage, we use the zero-electric-winding sector and project to the momentum component containing the exact cage. The branch followed here has

$$
(k_x,k_y)=\left(0,\frac{2\pi}{4}[2N\bmod4]\right).
$$

The local observable $Z_R$ does not preserve momentum. Therefore the correct second moment is obtained by projecting $Z_R^2$ itself,

$$
\langle Z_R^2\rangle_{k}=\langle E,k|P_k Z_R^2P_k|E,k\rangle,
$$

rather than squaring $P_kZ_RP_k$. This distinction is essential for a local witness in a symmetry-resolved ensemble.

In [ ]:
def fixed_width_microcanonical_point(repeats, *, phase=PEIERLS_REFERENCE_PHASE):
    raw_instance = product_unit_cell.with_couplings(
        coup_kin=np.exp(1.0j * phase),
        coup_pot=1.0,
    ).instantiate(int(repeats))
    finite_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=finite_model)
    t0 = time.perf_counter()
    print(f"  building {finite_model.lx}x{finite_model.ly} zero-winding sector ...", flush=True)
    build = finite_model.build(
        basis_solver="dfs",
        builder="bitmask",
        backend="scipy",
        sort_basis=True,
    )
    print(f"  basis dimension: {build.basis.n_states}", flush=True)
    configs = basis_configs_from_build_result(build)
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_report = diagnose_eigenpair(build.hamiltonian, cage)

    tx = square_qdm_basis_translation_permutation(finite_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(finite_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(repeats)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty),
        orders=(finite_model.lx, finite_model.ly),
        momentum_indices=(kx, ky),
        labels={
            "winding_x": 0,
            "winding_y": 0,
            "kx_index": kx,
            "ky_index": ky,
        },
    )
    cage_sector = project_state_to_sector(cage, sector)
    projection_norm = float(np.linalg.norm(cage_sector))
    if projection_norm <= TOL:
        raise RuntimeError("The predicted momentum branch has zero cage weight.")
    cage_sector /= projection_norm

    print(f"  momentum sector {(kx, ky)} dimension: {sector.sector_dimension}", flush=True)
    h_sector = project_operator_to_sector(build.hamiltonian, sector)
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)
    print("  diagonalization complete", flush=True)
    overlaps = np.abs(vectors.conj().T @ cage_sector) ** 2
    scar_index = int(np.argmax(overlaps))
    target_energy = float(cage_report.energy.real)
    degenerate_mask = np.abs(energies - target_energy) <= 1.0e-8

    witness = selected_strip_witness.placement.instantiate_on_model(finite_model)
    z_full = witness.embed(configs)
    q_full = z_full.conj().T @ z_full
    z_sector = project_operator_to_sector(z_full, sector)
    q_sector = project_operator_to_sector(q_full, sector)
    cage_q = float(np.vdot(cage_sector, q_sector @ cage_sector).real)

    print("  projected witness operators", flush=True)
    q_expectations = eigenstate_expectations(q_sector, vectors)
    z_expectations = eigenstate_expectations(z_sector, vectors)
    print("  eigenstate witness expectations complete", flush=True)
    rows = []
    primary = None
    for prefactor in MICROCANONICAL_PREFACTORS:
        plan = thermodynamic_energy_window_plan(
            volume=finite_model.lattice.num_plaquettes,
            energy_density=peierls_product_sequence.energy_density,
            width_prefactor=prefactor,
            local_energy_scale=1.0,
        )
        window = select_microcanonical_window_by_width(
            energies,
            target_energy=target_energy,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        moments = spectral_observable_moments(
            z_sector,
            vectors,
            squared_operator=q_sector,
            indices=window.indices,
        )
        row = {
            "repeats": int(repeats),
            "Lx": int(finite_model.lx),
            "Ly": int(finite_model.ly),
            "volume": int(finite_model.lattice.num_plaquettes),
            "winding_x": 0,
            "winding_y": 0,
            "kx_index": int(kx),
            "ky_index": int(ky),
            "full_winding_sector_dimension": int(configs.shape[0]),
            "resolved_sector_dimension": int(sector.sector_dimension),
            "Peierls_phase": float(phase),
            "cage_energy": target_energy,
            "cage_energy_density": target_energy / finite_model.lattice.num_plaquettes,
            "cage_residual": cage_report.residual_norm,
            "cage_projection_norm": projection_norm,
            "scar_max_overlap": float(overlaps[scar_index]),
            "scar_degenerate_subspace_weight": float(np.sum(overlaps[degenerate_mask])),
            "scar_degenerate_level_count": int(np.sum(degenerate_mask)),
            "scar_level_energy": float(energies[scar_index]),
            "scar_Q_expectation": cage_q,
            "window_prefactor": float(prefactor),
            "window_requested_half_width": plan.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "window_actual_half_width": window.half_width,
            "window_state_count": window.n_states,
            "window_center_offset": window.center_offset,
            "thermal_Z_mean": moments.mean,
            "thermal_Z_second_moment": moments.second_moment,
            "thermal_Z_variance": moments.variance,
            "runtime_seconds": time.perf_counter() - t0,
        }
        rows.append(row)
        if abs(prefactor - PRIMARY_WINDOW_PREFACTOR) <= TOL:
            primary = row
    if primary is None:
        raise RuntimeError("PRIMARY_WINDOW_PREFACTOR is absent from the sensitivity list.")

    smooth = gaussian_spectral_filter(
        energies,
        target_energy=target_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * np.sqrt(finite_model.lattice.num_plaquettes),
    )
    smooth_moments = spectral_observable_moments(
        z_sector,
        vectors,
        squared_operator=q_sector,
        weights=np.asarray(smooth.weights),
    )
    try:
        gap = adjacent_gap_ratio_report(
            energies,
            trim_fraction=0.10,
            degeneracy_tolerance=RANK_TOL,
        )
        mean_gap_ratio = gap.mean_ratio
        gap_ratio_count = len(gap.ratios)
    except ValueError:
        mean_gap_ratio = np.nan
        gap_ratio_count = 0

    primary.update({
        "smooth_Z_mean": smooth_moments.mean,
        "smooth_Z_second_moment": smooth_moments.second_moment,
        "smooth_Z_variance": smooth_moments.variance,
        "smooth_effective_state_count": smooth.effective_state_count,
        "sharp_smooth_second_moment_difference": abs(
            primary["thermal_Z_second_moment"] - smooth_moments.second_moment
        ),
        "mean_gap_ratio": mean_gap_ratio,
        "gap_ratio_count": gap_ratio_count,
    })
    print(f"  microcanonical point complete in {time.perf_counter() - t0:.2f} s", flush=True)
    scatter = pd.DataFrame({
        "repeats": int(repeats),
        "Lx": int(finite_model.lx),
        "energy": energies,
        "energy_density": energies / finite_model.lattice.num_plaquettes,
        "Z_mean": z_expectations,
        "Q_expectation": q_expectations,
        "is_scar_level": np.arange(energies.size) == scar_index,
    })
    return rows, primary, scatter


repeat_counts = (1, 2) if RUN_8X4_MICROCANONICAL else (1,)
fixed_width_rows = []
fixed_width_primary = []
fixed_width_scatter = []
for repeats in repeat_counts:
    rows, primary, scatter = fixed_width_microcanonical_point(repeats)
    fixed_width_rows.extend(rows)
    fixed_width_primary.append(primary)
    fixed_width_scatter.append(scatter)

fixed_width_window_table = pd.DataFrame(fixed_width_rows)
fixed_width_primary_table = pd.DataFrame(fixed_width_primary)
fixed_width_scatter_table = pd.concat(fixed_width_scatter, ignore_index=True)
fixed_width_window_table.to_csv(
    DATA_DIR / "qdm_fixed_width_microcanonical_window_sensitivity.csv",
    index=False,
)
fixed_width_primary_table.to_csv(
    DATA_DIR / "qdm_fixed_width_microcanonical_primary.csv",
    index=False,
)
fixed_width_scatter_table.to_csv(
    DATA_DIR / "qdm_fixed_width_eth_scatter.csv",
    index=False,
)
display(fixed_width_primary_table)

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.plot(
    fixed_width_primary_table["Lx"],
    fixed_width_primary_table["thermal_Z_second_moment"],
    marker="o",
    label=r"sharp $\langle Z_R^2\rangle_{\rm mc}$",
)
ax.plot(
    fixed_width_primary_table["Lx"],
    fixed_width_primary_table["thermal_Z_variance"],
    marker="s",
    label=r"thermal variance",
)
ax.plot(
    fixed_width_primary_table["Lx"],
    np.abs(fixed_width_primary_table["thermal_Z_mean"]),
    marker="^",
    label=r"$|\langle Z_R\rangle_{\rm mc}|$",
)
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage activity")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel("Normalized local moment")
ax.legend(frameon=False, fontsize=7)
save_figure(fig, "qdm_fixed_width_microcanonical_activity")
plt.show()

largest_Lx = int(fixed_width_scatter_table["Lx"].max())
scatter_largest = fixed_width_scatter_table[
    fixed_width_scatter_table["Lx"] == largest_Lx
]
fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.scatter(
    scatter_largest["energy_density"],
    scatter_largest["Q_expectation"],
    s=11,
    alpha=0.65,
    label="resolved-sector eigenstates",
)
scar_row = scatter_largest[scatter_largest["is_scar_level"]]
ax.scatter(
    scar_row["energy_density"],
    scar_row["Q_expectation"],
    marker="*",
    s=80,
    label="cage component",
)
ax.set_xlabel("Energy density")
ax.set_ylabel(r"$\langle Z_R^2\rangle$")
ax.legend(frameon=False, fontsize=7)
save_figure(fig, "qdm_fixed_width_eth_scatter_largest")
plt.show()

The primary finite-width comparison now satisfies the manuscript's ensemble requirements: the winding and momentum sector are recorded, the window is centered at the exact cage energy, the energy-density width scales as $L_x^{-1/2}$, complete degeneracies are retained, and the state count is reported. The cage remains exactly dark while the microcanonical second moment is positive on the accessible sizes.

This is still evidence rather than a completed thermodynamic proof. Width four may have integrable or fragmented regimes, the gap ratio can retain thin-torus symmetry effects, and two lengths do not establish a nonzero liminf. The notebook therefore reports the spectral statistic but does not use it as a substitute for the local-activity scaling.

## 8. Thermal-activity margin along a preserving Peierls path

The uniform plaquette phase preserves the compact cage and the same bounded witness. At $4\times4$ we follow the true sharp microcanonical activity and a Gaussian-filtered activity centered at the deformed cage energy. The smooth curve is used only for the finite-size susceptibility; the sharp window remains the ETH comparator.

In [ ]:
def peierls_thermal_point_4x4(phase):
    phase_model = replace(square_model, coup_kin=np.exp(1.0j * float(phase)))
    build = phase_model.build(
        basis_solver="dfs",
        builder="sparse",
        backend="scipy",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(build)
    tx = square_qdm_basis_translation_permutation(phase_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(phase_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(1)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty),
        orders=(4, 4),
        momentum_indices=(kx, ky),
    )
    cage_sector = project_state_to_sector(compact_state, sector)
    cage_sector /= np.linalg.norm(cage_sector)
    h_sector = project_operator_to_sector(build.hamiltonian, sector)
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)

    witness = selected_strip_witness.placement.instantiate_on_model(phase_model)
    z_full = witness.embed(configs)
    q_full = z_full.conj().T @ z_full
    z_sector = project_operator_to_sector(z_full, sector)
    q_sector = project_operator_to_sector(q_full, sector)
    cage_energy = float(diagnose_eigenpair(build.hamiltonian, compact_state).energy.real)
    plan = thermodynamic_energy_window_plan(
        volume=16,
        energy_density=cage_energy / 16.0,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=1.0,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=cage_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    sharp = spectral_observable_moments(
        z_sector,
        vectors,
        squared_operator=q_sector,
        indices=window.indices,
    )
    smooth_filter = gaussian_spectral_filter(
        energies,
        target_energy=cage_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * np.sqrt(16),
    )
    smooth = spectral_observable_moments(
        z_sector,
        vectors,
        squared_operator=q_sector,
        weights=np.asarray(smooth_filter.weights),
    )
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        build.hamiltonian,
        np.flatnonzero(np.abs(compact_state) > TOL),
        compact_state,
        tolerance=TOL,
    )
    return {
        "phase": float(phase),
        "path_parameter": float(phase - PEIERLS_REFERENCE_PHASE),
        "cage_energy": cage_energy,
        "cage_residual": diagnose_eigenpair(build.hamiltonian, compact_state).residual_norm,
        "Delta_cage": conditioning.cage_gap,
        "Delta_Q": local_q_spectrum.dark_channel_gap,
        "window_state_count": window.n_states,
        "window_energy_density_half_width": plan.energy_density_half_width,
        "sharp_Z_mean": sharp.mean,
        "sharp_Q_activity": sharp.second_moment,
        "sharp_Z_variance": sharp.variance,
        "smooth_Q_activity": smooth.second_moment,
        "smooth_effective_state_count": smooth_filter.effective_state_count,
        "sharp_smooth_difference": abs(sharp.second_moment - smooth.second_moment),
    }


peierls_thermal_table = pd.DataFrame([
    peierls_thermal_point_4x4(phase)
    for phase in PEIERLS_PATH
])
peierls_thermal_margin = thermal_activity_margin_from_samples(
    peierls_thermal_table["path_parameter"],
    peierls_thermal_table["smooth_Q_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
peierls_thermal_table.to_csv(
    DATA_DIR / "qdm_4x4_peierls_thermal_margin_path.csv",
    index=False,
)
pd.DataFrame([peierls_thermal_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "qdm_4x4_peierls_thermal_margin.csv",
    index=False,
)
display(peierls_thermal_table)
display(pd.DataFrame([peierls_thermal_margin.to_summary_dict()]))

fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.plot(
    peierls_thermal_table["phase"],
    peierls_thermal_table["sharp_Q_activity"],
    marker="o",
    label="sharp microcanonical",
)
ax.plot(
    peierls_thermal_table["phase"],
    peierls_thermal_table["smooth_Q_activity"],
    marker="s",
    label="smooth filter",
)
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel(r"Thermal $\langle Z_R^2\rangle$")
ax.legend(frameon=False, fontsize=7)
save_figure(fig, "qdm_peierls_thermal_margin")
plt.show()

fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.plot(
    peierls_thermal_table["phase"],
    peierls_thermal_table["Delta_cage"],
    marker="o",
    label=r"$\Delta_{\rm cage}$",
)
ax.axhline(local_q_spectrum.dark_channel_gap, linestyle="--", label=r"$\Delta_Q$")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Conditioning scale")
ax.legend(frameon=False)
save_figure(fig, "qdm_peierls_conditioning_and_local_gap")
plt.show()

## 9. Energy-density matching and the pure-kinetic trap

For the uniform RK potential, the repeated cage has $e_{\rm cage}=1/4$, while the width-four beta-zero energy density approaches a slightly larger value.  Thus the uniform model requires a finite-temperature comparison.  Setting the potential coupling to zero gives exact beta-zero matching, but produces a large chiral zero-mode manifold on the finite torus and contaminates level statistics near the cage energy.

In [ ]:
energy_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256)
beta_zero_energy_report = scan_square_qdm_beta_zero_energy_density(
    tuple((length, 4) for length in energy_lengths),
    potential_coupling=1.0,
    winding_sector=(0, 0),
    winding_projection="fourier",
)
energy_table = pd.DataFrame([
    {
        "length": evaluation.length,
        "circumference": evaluation.circumference,
        "beta_zero_energy_density": evaluation.energy_density,
        "uniform_cage_energy_density": product_sequence.energy_density,
        "mismatch": evaluation.energy_density - product_sequence.energy_density,
    }
    for evaluation in beta_zero_energy_report.evaluations
])
energy_table.to_csv(DATA_DIR / "qdm_strip_beta_zero_energy_density.csv", index=False)
display(energy_table)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(energy_table["length"], energy_table["beta_zero_energy_density"], marker="o", label=r"$e_{\beta=0}$")
ax.axhline(product_sequence.energy_density, linestyle="--", linewidth=0.8, label=r"uniform cage $e=1/4$")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel("Energy density")
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_energy_matching")
plt.show()

In [ ]:
kinetic_eigenvalues = scipy_linalg.eigvalsh(square_build.kinetic.toarray())
kinetic_zero_count = int(np.sum(np.abs(kinetic_eigenvalues) <= RANK_TOL))
kinetic_zero_fraction = kinetic_zero_count / kinetic_eigenvalues.size
kinetic_gap_report = adjacent_gap_ratio_report(
    kinetic_eigenvalues,
    trim_fraction=0.1,
    degeneracy_tolerance=RANK_TOL,
)
{
    "pure_kinetic_beta_zero_match": True,
    "zero_mode_count_4x4_w00": kinetic_zero_count,
    "zero_mode_fraction": kinetic_zero_fraction,
    "usable_gap_ratios_after_degeneracy_filter": len(kinetic_gap_report.ratios),
    "mean_adjacent_gap_ratio": kinetic_gap_report.mean_ratio,
}

The pure-kinetic point is therefore useful for an exact energy-density identity but poor as the sole thermal-background demonstration.  A finite or inhomogeneous diagonal term is needed to lift accidental zero modes while preserving the cage.

## 10. Finite-size beta-zero-matched inhomogeneous control

Every plaquette flippability projector has a definite eigenvalue on the compact cage.  Therefore arbitrary plaquette-dependent potential coefficients preserve the exact state.  On the $4\times4$ torus, a single linear constraint matches its energy to the beta-zero trace.  We project a documented random coefficient vector onto this 15-dimensional matching space.  The inhomogeneity breaks translations and point-group symmetries and removes the pure-kinetic zero manifold.

This is a finite-size thermal-background control, not yet a full strip deformation theorem.

In [ ]:
if len(potential_term_matrices) != 2 * len(square_model.plaquette_ids()):
    raise RuntimeError("Expected two orientation projectors per square plaquette.")
plaquette_potential_matrices = tuple(
    potential_term_matrices[2 * index] + potential_term_matrices[2 * index + 1]
    for index in range(len(square_model.plaquette_ids()))
)

compact_state = states_04[:, 0]
scar_flippabilities = np.asarray([
    np.vdot(compact_state, matrix @ compact_state).real
    for matrix in plaquette_potential_matrices
])
finite_beta_zero_flippabilities = np.asarray([
    np.mean(matrix.diagonal().real)
    for matrix in plaquette_potential_matrices
])
finite_match = beta_zero_matching_subspace(
    scar_flippabilities,
    finite_beta_zero_flippabilities,
    tolerance=TOL,
)

rng = np.random.default_rng(RANDOM_SEED)
matched_coefficients = project_coefficients_to_beta_zero_match(
    rng.normal(size=len(plaquette_potential_matrices)),
    finite_match,
    normalize=True,
) * 4.0

matched_hamiltonian = square_build.kinetic.astype(np.complex128)
for coefficient, matrix in zip(matched_coefficients, plaquette_potential_matrices, strict=True):
    matched_hamiltonian = matched_hamiltonian + coefficient * matrix

matched_eigenvalues, matched_eigenvectors = scipy_linalg.eigh(matched_hamiltonian.toarray())
matched_scar_report = diagnose_eigenpair(matched_hamiltonian, compact_state)
matched_gap_report = adjacent_gap_ratio_report(
    matched_eigenvalues,
    trim_fraction=0.1,
    degeneracy_tolerance=RANK_TOL,
)

full_classification = classify_cage_state(
    records_04[0].cage_state,
    kinetic_matrix=square_build.kinetic,
    basis_configs=square_build.basis.states,
    hilbert_size=square_search.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
full_witness = local_witnesses_from_classification_report(full_classification)[0]
normalized_template = full_witness.template.normalized("operator_norm")
normalized_witness = normalized_template.instantiate(full_witness.variable_indices)
local_row_operator = normalized_witness.embed(square_build.basis.states)
q_operator = (local_row_operator.conj().T @ local_row_operator).toarray()

scar_overlaps = np.abs(matched_eigenvectors.conj().T @ compact_state) ** 2
scar_index = int(np.argmax(scar_overlaps))
q_expectations = eigenstate_expectations(q_operator, matched_eigenvectors)
window = select_microcanonical_window_by_count(
    matched_eigenvalues,
    target_energy=float(matched_scar_report.energy.real),
    target_count=24,
    exclude_indices=(scar_index,),
    include_boundary_degeneracy=True,
    degeneracy_tolerance=TOL,
)
window_indices = np.asarray(window.indices, dtype=np.int64)

matched_summary = {
    **finite_match.to_summary_dict(),
    "scar_energy": float(matched_scar_report.energy.real),
    "beta_zero_trace_energy": float(np.trace(matched_hamiltonian.toarray()).real / square_search.hilbert_size),
    "scar_residual": matched_scar_report.residual_norm,
    "scar_overlap": float(scar_overlaps[scar_index]),
    "zero_mode_count": int(np.sum(np.abs(matched_eigenvalues) <= RANK_TOL)),
    "mean_gap_ratio": matched_gap_report.mean_ratio,
    "microcanonical_state_count": window.n_states,
    "microcanonical_half_width": window.half_width,
    "microcanonical_center_offset": window.center_offset,
    "microcanonical_q_mean": float(np.mean(q_expectations[window_indices])),
    "microcanonical_q_minimum": float(np.min(q_expectations[window_indices])),
    "scar_q_expectation": float(q_expectations[scar_index]),
}
pd.DataFrame([matched_summary]).to_csv(DATA_DIR / "qdm_4x4_beta_zero_matched_control.csv", index=False)
pd.DataFrame({
    "plaquette": np.arange(len(matched_coefficients)),
    "coefficient": matched_coefficients,
    "scar_flippability": scar_flippabilities,
    "finite_beta_zero_flippability": finite_beta_zero_flippabilities,
}).to_csv(DATA_DIR / "qdm_4x4_beta_zero_matched_coefficients.csv", index=False)
matched_summary

In [ ]:
spectral_table = pd.DataFrame({
    "energy": matched_eigenvalues,
    "energy_density": matched_eigenvalues / 16.0,
    "q_expectation": q_expectations,
    "is_scar": np.arange(matched_eigenvalues.size) == scar_index,
    "is_microcanonical": np.isin(np.arange(matched_eigenvalues.size), window_indices),
})
spectral_table.to_csv(DATA_DIR / "qdm_4x4_matched_eth_scatter.csv", index=False)

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.scatter(spectral_table["energy_density"], spectral_table["q_expectation"], s=10, alpha=0.65, label="sector eigenstates")
ax.scatter(
    [spectral_table.loc[scar_index, "energy_density"]],
    [spectral_table.loc[scar_index, "q_expectation"]],
    marker="*", s=75, label="compact cage",
)
ax.set_xlabel("Energy density")
ax.set_ylabel(r"$\langle Q_R\rangle$")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_matched_eth_scatter")
plt.show()

For this documented finite-size control, the exact cage sits at the beta-zero trace energy, has zero local-witness activity, and is surrounded by states with positive $\langle Q_R\rangle$.  The mean adjacent-gap ratio is closer to GOE than to Poisson, but the Hilbert space contains only 132 states.  The result should be reported as corroborating finite-size evidence, not as a thermodynamic chaos proof.

## 11. Quasi-one-dimensional audit

The following report makes the limitations machine-readable.  It combines exact sequence data, fixed-width witness activity, the linearly growing compatibility rank, the uniform-model energy mismatch, and the pure-kinetic zero-mode problem.

In [ ]:
primary_by_length = {
    int(row.Lx): row
    for row in fixed_width_primary_table.itertuples(index=False)
}
scaling_points = []
for point in product_scaling.points:
    length = int(point.system_size[0])
    primary = primary_by_length.get(length)
    if primary is None:
        continue
    scaling_points.append(Quasi1DSequencePoint(
        length=length,
        width=int(point.system_size[1]),
        exact_residual=float(point.product_state_boundary_residual),
        witness_radius=float(selected_strip_witness.placement.window_width),
        transverse_witness_span=float(selected_strip_witness.placement.circumference),
        thermal_second_moment=float(primary.thermal_Z_second_moment),
        interference_gap=float(point.interference_gap),
        compatibility_rank=int(point.kinetic_compatibility.rank),
        local_parameter_count=16 * int(point.repeats),
        support_size=int(point.support_size),
        sector_dimension=int(primary.resolved_sector_dimension),
    ))

# The primary ensemble is centered at the exact cage energy, so its energy
# mismatch is zero by construction. The pure-kinetic zero-mode issue is kept
# in Sec. 9 as a warning about a different Hamiltonian, not attributed to the
# reference Peierls-deformed microcanonical calculation.
level_gap_ratio = float(fixed_width_primary_table.iloc[-1]["mean_gap_ratio"])
quasi_1d_audit = audit_quasi_1d_sequence(
    scaling_points,
    energy_density_mismatch=0.0,
    thermal_comparison="energy_matched_microcanonical",
    level_gap_ratio=level_gap_ratio,
    zero_mode_fraction=0.0,
    tolerance=1.0e-8,
)

pd.DataFrame([point.to_summary_dict() for point in scaling_points]).to_csv(
    DATA_DIR / "qdm_quasi_1d_audit_points.csv", index=False
)
pd.DataFrame({
    "kind": ["established"] * len(quasi_1d_audit.established)
        + ["problem"] * len(quasi_1d_audit.issues),
    "statement": list(quasi_1d_audit.established) + list(quasi_1d_audit.issues),
}).to_csv(DATA_DIR / "qdm_quasi_1d_audit_statements.csv", index=False)

print("Established:")
for statement in quasi_1d_audit.established:
    print("  +", statement)
print("\nProblems / limitations:")
for statement in quasi_1d_audit.issues:
    print("  -", statement)

### Interpretation and remaining fixed-width caveats

The fixed-width calculation supports a conservative statement:

> There is an exact $(4N)\times4$ square-QDM cage sequence in the zero-electric-winding sector, with a size-independent Hermitian reduced-IZ witness. In the momentum sector containing the cage, an energy-matched microcanonical window of width $\Delta E\propto\sqrt{4L_x}$ has a positive local second moment on the accessible sizes.

The following qualifications remain essential.

- **Not a two-dimensional limit.** Width four defines an effective one-dimensional constrained model. Its phases and ETH behavior need not represent the square-QDM plane.
- **The sector is part of the definition.** Winding is fixed before diagonalization. For the translation-invariant Peierls model, momentum is also resolved; the relevant momentum branch alternates with the repeat parity.
- **The witness is only quasi-1D local.** Its support is bounded as $L_x\to\infty$, but it occupies a finite fraction of the circumference. Nothing here establishes locality under $L_y\to\infty$.
- **Window scaling and state count are separate tests.** $\Delta E/|\Omega|\to0$ follows by construction, whereas growth of the retained state count must be checked from the spectrum.
- **A positive activity is not by itself a chaos proof.** Thin-torus integrability, residual symmetry, or fragmentation can persist. The reported adjacent-gap ratio is a diagnostic, not a premise of the local-witness calculation.
- **Peierls phases change the symmetry class.** A generic uniform phase breaks ordinary time reversal, but residual unitary or antiunitary symmetries may remain. A GUE comparison is appropriate only after those are fully resolved.
- **Protection remains extensively constrained.** The repeated product cage needs two local kinetic compatibility conditions per unit cell. Exact size extensibility is therefore weaker than finite-codimension robustness.
- **The collective $4\times4$ quotient does not continue here.** Its fixed-width completeness defect disappears on the tested larger strips, and the uniform Peierls path lifts the selected quotient state.

The beta-zero transfer activity remains useful as a long-strip reference and as a check that the winding constraint does not force the witness to vanish. The primary ETH comparison is nevertheless the energy-matched microcanonical ensemble above.

## 12. Optional expensive collective-extension scan

The codebase contains a finite-range column-grammar search that asks whether the collective $4\times4$ record extends to a nonfactorized width-four cage outside the translated product span.  It can be much more expensive than the main notebook and is therefore disabled by default.  A positive `locality_extension_index` would be genuinely new; a zero result is only a no-go within the tested grammar.

In [ ]:
RUN_EXPENSIVE_COLLECTIVE_EXTENSION = False

if RUN_EXPENSIVE_COLLECTIVE_EXTENSION:
    collective_record = records_04[8]
    collective_support_configs = np.asarray(
        [square_build.basis.state(int(index)) for index in collective_record.cage_state.support],
        dtype=np.int64,
    )
    collective_extension = scan_square_qdm_collective_locality_extension(
        square_model,
        collective_support_configs,
        product_unit_cell,
        cases=((2, 8), (3, 8), (3, 12)),
        potential_per_column=1.0,
        max_words=10_000,
        max_product_support_size=128,
        dense_column_limit=512,
        maximum_nullity=16,
        ipr_restarts=32,
        tolerance=1.0e-9,
    )
    collective_extension_table = pd.DataFrame([
        point.to_summary_dict() for point in collective_extension.points
    ])
    collective_extension_table.to_csv(
        DATA_DIR / "qdm_collective_locality_extension.csv", index=False
    )
    display(collective_extension_table)
else:
    print("Skipped. Set RUN_EXPENSIVE_COLLECTIVE_EXTENSION=True for the finite-range grammar scan.")

## 12. State-resolved deformation spectra for the compact and collective representatives

In [ ]:
state_resolved_reports = {}
state_resolved_rows = []
state_resolved_singular_rows = []
state_operator_basis = kinetic_term_matrices + phase_tangent_matrices
for label, state in (("compact", compact_state), ("collective", collective_state)):
    report = operator_coefficient_compatibility(
        state_operator_basis,
        state,
        mode="fixed_vectors",
        tolerance=RANK_TOL,
    )
    state_resolved_reports[label] = report
    state_resolved_rows.append(
        {
            "target": label,
            "n_operators": report.n_operators,
            "compatible_dimension": report.compatible_dimension,
            "obstruction_rank": report.rank,
            "singular_gap": report.singular_gap,
        }
    )
    state_resolved_singular_rows.extend(
        {
            "target": label,
            "singular_index": int(i),
            "singular_value": float(value),
        }
        for i, value in enumerate(report.singular_values)
    )

state_resolved_table = pd.DataFrame(state_resolved_rows)
state_resolved_spectrum_table = pd.DataFrame(state_resolved_singular_rows)
state_resolved_table.to_csv(DATA_DIR / "qdm_state_resolved_preserving_dimensions.csv", index=False)
state_resolved_spectrum_table.to_csv(DATA_DIR / "qdm_state_resolved_singular_spectra.csv", index=False)
display(state_resolved_table)

fig, ax = plt.subplots(figsize=(3.35, 2.45))
for label, marker in (("compact", "o"), ("collective", "s")):
    subset = state_resolved_spectrum_table[state_resolved_spectrum_table["target"] == label]
    ax.semilogy(subset["singular_index"], subset["singular_value"], marker=marker, label=label)
ax.set_xlabel("Singular-value index")
ax.set_ylabel("State-resolved obstruction spectrum")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_state_resolved_singular_spectra")
plt.show()


## 13. Draft-ready composite figures for Sec. 7

In [ ]:
fixed_width_bands = fixed_width_window_table.groupby("Lx")
fixed_width_panel = fixed_width_primary_table.copy()
fixed_width_panel["thermal_low"] = fixed_width_bands["thermal_Z_second_moment"].min().reindex(fixed_width_panel["Lx"]).to_numpy()
fixed_width_panel["thermal_high"] = fixed_width_bands["thermal_Z_second_moment"].max().reindex(fixed_width_panel["Lx"]).to_numpy()
fixed_width_panel["mean_abs"] = np.abs(fixed_width_panel["thermal_Z_mean"])

fig, ax = plt.subplots(figsize=(3.35, 2.45))
values = fixed_width_panel["thermal_Z_second_moment"].to_numpy()
yerr = np.vstack([
    values - fixed_width_panel["thermal_low"].to_numpy(),
    fixed_width_panel["thermal_high"].to_numpy() - values,
])
ax.errorbar(
    fixed_width_panel["Lx"],
    values,
    yerr=yerr,
    marker="o",
    capsize=3,
    label=r"$\langle \widetilde Q_R^Z\rangle_{\rm mc}$",
)
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage value")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel(r"Thermal $\langle \widetilde Q_R^Z\rangle$")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_fixed_width_thermal_scaling")
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["sharp_Q_activity"], marker="o", label=r"$\langle \widetilde Q_R^Z\rangle_{\rm mc}$")
ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["sharp_Z_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(\widetilde Z_R)$")
ax.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["sharp_Z_mean"]), marker="^", label=r"$|\langle \widetilde Z_R\rangle_{\rm mc}|$")
ax.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Witness moment")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_witness_resolution_peierls_scan")
plt.show()

fig = plt.figure(figsize=(7.0, 3.0))
grid = fig.add_gridspec(1, 2, wspace=0.30)
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])
values = fixed_width_panel["thermal_Z_second_moment"].to_numpy()
yerr = np.vstack([
    values - fixed_width_panel["thermal_low"].to_numpy(),
    fixed_width_panel["thermal_high"].to_numpy() - values,
])
ax0.errorbar(
    fixed_width_panel["Lx"],
    values,
    yerr=yerr,
    marker="o",
    capsize=3,
    label=r"$\langle \widetilde Q_R^Z\rangle_{\rm mc}$",
)
ax0.axhline(0.0, linestyle="--", linewidth=0.8, label="cage value")
ax0.set_xlabel(r"Strip length $L_x$")
ax0.set_ylabel(r"Thermal $\langle \widetilde Q_R^Z\rangle$")
ax0.grid(alpha=0.3)
ax0.legend(frameon=False, fontsize=6)
ax0.text(0.02, 0.98, "(a)", transform=ax0.transAxes, ha="left", va="top")
ax1.plot(peierls_thermal_table["phase"], peierls_thermal_table["sharp_Q_activity"], marker="o", label=r"$\langle \widetilde Q_R^Z\rangle_{\rm mc}$")
ax1.plot(peierls_thermal_table["phase"], peierls_thermal_table["sharp_Z_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(\widetilde Z_R)$")
ax1.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["sharp_Z_mean"]), marker="^", label=r"$|\langle \widetilde Z_R\rangle_{\rm mc}|$")
ax1.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax1.set_xlabel(r"Uniform plaquette phase $\phi$")
ax1.set_ylabel("Witness moment")
ax1.grid(alpha=0.3)
ax1.legend(frameon=False, fontsize=6)
ax1.text(0.02, 0.98, "(b)", transform=ax1.transAxes, ha="left", va="top")
fig.tight_layout()
save_figure(fig, "qdm_missing_fixed_width_deformation_data")
plt.show()


## Export manifest

In [ ]:
manifest = sorted(
    str(path.relative_to(REPO_ROOT))
    for path in DATA_DIR.rglob("*")
    if path.is_file()
)
for item in manifest:
    print(item)